# Assignment 3 — Handwritten Digit Generation with DCGAN

Implementing a Deep Convolutional GAN (DCGAN) on the MNIST dataset.  
The generator learns to produce realistic handwritten digits from random noise; the discriminator learns to tell them apart from real ones.

**Reference:** Radford, Metz, Chintala (2015) — *Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks*, arXiv:1511.06434

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# --- Hyperparameters ---
LATENT_DIM = 100     # size of the input noise vector
BATCH_SIZE = 128
EPOCHS     = 50
LR         = 0.0002  # learning rate from DCGAN paper
BETA1      = 0.5     # Adam beta1 from DCGAN paper
SEED       = 42

torch.manual_seed(SEED)

## Dataset

MNIST images are 28×28 grayscale, values 0–255.  
We normalize to [−1, 1] to match the Generator's `Tanh` output.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])   # (x - 0.5) / 0.5  →  [-1, 1]
])

dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
# num_workers=0 is safer inside Jupyter on Windows
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f'Training images: {len(dataset)}')
print(f'Batches per epoch: {len(dataloader)}')

In [ ]:
# Quick sanity check — show a few real images
sample_batch, _ = next(iter(dataloader))
fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for ax, img in zip(axes, sample_batch[:8]):
    ax.imshow(img.squeeze(), cmap='gray', vmin=-1, vmax=1)
    ax.axis('off')
plt.suptitle('Sample real images')
plt.tight_layout()
plt.show()

## Architecture

### Generator
Takes a 100-d noise vector and upsamples to a 28×28 image:
```
Linear(100 → 128×7×7)  →  reshape(128, 7, 7)
ConvTranspose2d(128→64, k=4, s=2, p=1)   # 7  → 14
BatchNorm + ReLU
ConvTranspose2d(64→1,  k=4, s=2, p=1)    # 14 → 28
Tanh
```

### Discriminator
Takes a 28×28 image and outputs a real/fake probability:
```
Conv2d(1→64,   k=4, s=2, p=1)   # 28 → 14
LeakyReLU(0.2)
Conv2d(64→128, k=4, s=2, p=1)   # 14 → 7
BatchNorm + LeakyReLU(0.2)
Linear(128×7×7 → 1)  +  Sigmoid
```

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 7 * 7)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # 7→14
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),    # 14→28
            nn.Tanh()
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 128, 7, 7)
        return self.net(x)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),    # 28→14
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # 14→7
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.fc = nn.Linear(128 * 7 * 7, 1)

    def forward(self, x):
        features = self.net(x).view(-1, 128 * 7 * 7)
        return torch.sigmoid(self.fc(features))

In [ ]:
def weights_init(m):
    """From the DCGAN paper: init Conv and BN weights from N(0, 0.02)."""
    cls = m.__class__.__name__
    if 'Conv' in cls:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cls:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

G = Generator(LATENT_DIM).to(device)
D = Discriminator().to(device)
G.apply(weights_init)
D.apply(weights_init)

print(G)
print()
print(D)

total_g = sum(p.numel() for p in G.parameters())
total_d = sum(p.numel() for p in D.parameters())
print(f'\nGenerator params:     {total_g:,}')
print(f'Discriminator params: {total_d:,}')

In [ ]:
criterion = nn.BCELoss()

G_optim = torch.optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
D_optim = torch.optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))

# fixed noise so we can watch the same latent points evolve over training
fixed_noise = torch.randn(16, LATENT_DIM, device=device)

## Training

Each iteration:
1. **Train D** on a real batch (target = 1) and a fake batch (target = 0).
2. **Train G** using the same fake batch — but now the target is 1, so G gets gradient signal to fool D better.

`fake_imgs.detach()` is used when training D to avoid computing gradients through G.

In [ ]:
os.makedirs('samples', exist_ok=True)
d_losses, g_losses = [], []

for epoch in range(EPOCHS):
    d_run, g_run = 0.0, 0.0

    for real_imgs, _ in dataloader:
        real_imgs = real_imgs.to(device)
        bs = real_imgs.size(0)

        real_labels = torch.ones(bs, 1, device=device)
        fake_labels = torch.zeros(bs, 1, device=device)

        # ── Discriminator ─────────────────────────────────────────────
        D_optim.zero_grad()

        loss_real = criterion(D(real_imgs), real_labels)

        z = torch.randn(bs, LATENT_DIM, device=device)
        fake_imgs = G(z)
        loss_fake = criterion(D(fake_imgs.detach()), fake_labels)

        d_loss = loss_real + loss_fake
        d_loss.backward()
        D_optim.step()

        # ── Generator ─────────────────────────────────────────────────
        G_optim.zero_grad()

        # G wants D to output 1 (real) for its fake images
        g_loss = criterion(D(fake_imgs), real_labels)
        g_loss.backward()
        G_optim.step()

        d_run += d_loss.item()
        g_run += g_loss.item()

    avg_d = d_run / len(dataloader)
    avg_g = g_run / len(dataloader)
    d_losses.append(avg_d)
    g_losses.append(avg_g)

    print(f'Epoch [{epoch+1:>2}/{EPOCHS}]   D loss: {avg_d:.4f}   G loss: {avg_g:.4f}')

    # save a sample grid every 10 epochs
    if (epoch + 1) % 10 == 0:
        G.eval()
        with torch.no_grad():
            imgs = G(fixed_noise).cpu()
        G.train()

        fig, axes = plt.subplots(4, 4, figsize=(6, 6))
        for ax, img in zip(axes.flat, imgs):
            ax.imshow(img.squeeze(), cmap='gray', vmin=-1, vmax=1)
            ax.axis('off')
        plt.suptitle(f'Generated — Epoch {epoch+1}', fontsize=12)
        plt.tight_layout()
        plt.savefig(f'samples/epoch_{epoch+1:02d}.png', bbox_inches='tight')
        plt.show()

print('Training complete.')

## Results

In [ ]:
# Training loss curves
plt.figure(figsize=(8, 4))
plt.plot(d_losses, label='Discriminator')
plt.plot(g_losses, label='Generator')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('DCGAN Training Losses')
plt.legend()
plt.tight_layout()
plt.savefig('samples/loss_curve.png')
plt.show()

In [ ]:
# Generate 25 samples from the final model
G.eval()
with torch.no_grad():
    z_final = torch.randn(25, LATENT_DIM, device=device)
    final_imgs = G(z_final).cpu()

fig, axes = plt.subplots(5, 5, figsize=(7, 7))
for ax, img in zip(axes.flat, final_imgs):
    ax.imshow(img.squeeze(), cmap='gray', vmin=-1, vmax=1)
    ax.axis('off')
plt.suptitle('25 Generated Handwritten Digits (Final Model)', fontsize=12)
plt.tight_layout()
plt.savefig('samples/final_generated.png', bbox_inches='tight')
plt.show()

In [ ]:
# Save model weights
torch.save(G.state_dict(), 'samples/generator.pth')
torch.save(D.state_dict(), 'samples/discriminator.pth')
print('Saved weights to samples/')

## Discussion

**What worked:**
- The DCGAN paper's recommendations (BatchNorm, LeakyReLU 0.2, Adam with β₁=0.5) made training noticeably more stable compared to vanilla GAN.
- Normalizing MNIST to [−1, 1] to match Tanh is important — mismatched ranges slow down or prevent convergence.
- 50 epochs is enough for MNIST; digits are recognizable by around epoch 20.

**Common failure modes observed in GANs:**
- **Mode collapse**: G produces only a few digit types. Can be mitigated with minibatch discrimination or reducing G's learning rate.
- **D too strong too early**: G gets no useful gradient. Solved here by training both for exactly one step per batch.
- **Loss not converging to a fixed value**: In GANs this is normal — both losses oscillate as the two networks play against each other.

**Takeaway:** The adversarial loss pushes the generator toward the real data distribution without ever having an explicit reconstruction objective. The discriminator acts as an adaptive, learned loss function.